# 00 — Prueba de Spark (cierre de la Fase 0)

**Objetivo:** verificar que el entorno está listo para la Capa Bronce.

## 1. Raíz del proyecto e imports

In [2]:
import sys
from pathlib import Path

# Ubicar la raíz y hacer importable src/
RAIZ = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "ARQUITECTURA.md").exists())
if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))

from src.config import cargar_config, cargar_fuentes, cargar_reglas, ruta

print("Python :", sys.executable)
print("Raíz   :", RAIZ)

Python : /home/rodri/proyectos/Final_BI/.venv/bin/python
Raíz   : /home/rodri/proyectos/Final_BI


## 2. Configuración

In [3]:
cfg = cargar_config()
fuentes = cargar_fuentes()
reglas = cargar_reglas()

print("Rutas:")
for clave in cfg["rutas"]:
    print(f"  {clave:<11} -> {ruta(clave).relative_to(RAIZ)}")

print("\nFuentes:")
for fid, f in fuentes.items():
    destino = f.get("carpeta_bronze", "(no se ingiere)")
    print(f"  {fid:<4} {f['tipo']:<10} {destino:<28} {f['nombre']}")

print(f"\nReglas de calidad catalogadas: {len(reglas)}")
print("Spark:", cfg["spark"]["master"], "| driver_memory:", cfg["spark"]["driver_memory"])

Rutas:
  bronze      -> data/bronze
  silver      -> data/silver
  gold        -> data/gold
  quarantine  -> data/quarantine
  control     -> data/control
  muestras    -> data/_muestras
  manifest    -> data/bronze/manifest.json
  docs        -> docs

Fuentes:
  F1   descarga   siaf_ingreso                 SIAF — Presupuesto y ejecución de ingreso
  F2   descarga   sismepre_predial             SISMEPRE — Seguimiento de la meta del impuesto predial
  F3   descarga   renamu                       RENAMU (INEI) — ediciones 2009 a 2026

Reglas de calidad catalogadas: 19
Spark: local[*] | driver_memory: 4g


## 3. Sesión de Spark

In [4]:
from src.spark import crear_sesion

spark = crear_sesion("fase0_prueba")
sc = spark.sparkContext

print("Spark       :", spark.version)
print("Master      :", sc.master)
print("Paralelismo :", sc.defaultParallelism)
print("Java        :", sc._jvm.System.getProperty("java.version"))
print("UI          :", sc.uiWebUrl)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/11/20 19:22:27 WARN Utils: Your hostname, Rodrigo, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/11/20 19:22:27 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/20 19:22:28 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark       : 4.3.0
Master      : local[*]
Paralelismo : 12
Java        : 21.0.13.1
UI          : http://10.255.255.254:4040


## 4. Lectura de un CSV pequeño

Se lee **sin inferir tipos** (todo como texto), igual que se hará en el perfilado de Bronze.

In [5]:
csv_prueba = ruta("muestras") / "prueba_fase0.csv"

df = (
    spark.read
    .option("header", True)
    .option("inferSchema", False)
    .option("encoding", "UTF-8")
    .csv(str(csv_prueba))
)

df.printSchema()
df.show(truncate=False)

n_filas = df.count()
assert n_filas == 5, f"Se esperaban 5 filas, hay {n_filas}"
assert all(t == "string" for _, t in df.dtypes), "Sin inferSchema todas las columnas deben ser string"
assert df.filter(df.ubigeo == "040101").count() == 1, "El UBIGEO debe conservar el cero a la izquierda"
assert df.filter(df.municipalidad.contains("Ñ")).count() == 1, "Fallo de encoding UTF-8"
print("Lectura CSV: OK")

root
 |-- ubigeo: string (nullable = true)
 |-- municipalidad: string (nullable = true)
 |-- departamento: string (nullable = true)
 |-- anio: string (nullable = true)
 |-- monto_pia: string (nullable = true)
 |-- monto_recaudado: string (nullable = true)

+------+---------------------------------------------------+------------+----+----------+---------------+
|ubigeo|municipalidad                                      |departamento|anio|monto_pia |monto_recaudado|
+------+---------------------------------------------------+------------+----+----------+---------------+
|150101|Municipalidad de Prueba Uno                        |LIMA        |2024|1500000.50|1320000.00     |
|040101|Municipalidad de Prueba Dos                        |AREQUIPA    |2024|820000.00 |790500.25      |
|080101|Municipalidad de Prueba Tres                       |CUSCO       |2024|640000.00 |NULL           |
|130101|Municipalidad de Prueba Cuatro                     |LA LIBERTAD |2024|910000.75 |915000.00      |
|

## 5. Escritura y lectura de Parquet

In [6]:
import shutil
from pyspark.sql import functions as F

destino = ruta("muestras") / "_prueba_fase0_parquet"   # temporal: se borra al final de la celda

(df.withColumn("monto_pia", F.col("monto_pia").cast("decimal(18,2)"))
   .withColumn("monto_recaudado", F.col("monto_recaudado").cast("decimal(18,2)"))
   .write.mode("overwrite").parquet(str(destino)))

df_pq = spark.read.parquet(str(destino))
df_pq.printSchema()
total_pia = df_pq.agg(F.sum("monto_pia")).first()[0]
print("Suma monto_pia:", total_pia)

assert df_pq.count() == n_filas
assert str(total_pia) == "4170001.25"
shutil.rmtree(destino)   # artefacto temporal de la prueba
print("Parquet: OK")

26/11/20 19:22:41 WARN DataSource: All paths were ignored:
  file:/home/rodri/proyectos/Final_BI/data/control/_prueba_fase0_parquet


root
 |-- ubigeo: string (nullable = true)
 |-- municipalidad: string (nullable = true)
 |-- departamento: string (nullable = true)
 |-- anio: string (nullable = true)
 |-- monto_pia: decimal(18,2) (nullable = true)
 |-- monto_recaudado: decimal(18,2) (nullable = true)

Suma monto_pia: 4170001.25
Parquet: OK


## 6. Registro del resultado

In [7]:
import json, platform
from datetime import datetime, timezone

import pandas, pyarrow, pyspark

resultado = {
    "fase": 0,
    "verificacion": "sesion Spark + lectura CSV + escritura/lectura Parquet",
    "estado": "OK",
    "fecha_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "entorno": {
        "python": platform.python_version(),
        "java": sc._jvm.System.getProperty("java.version"),
        "pyspark": pyspark.__version__,
        "pandas": pandas.__version__,
        "pyarrow": pyarrow.__version__,
        "spark_master": sc.master,
        "paralelismo": sc.defaultParallelism,
        "driver_memory": spark.conf.get("spark.driver.memory"),
    },
    "csv_prueba": {"archivo": str(csv_prueba.relative_to(RAIZ)), "filas": n_filas, "columnas": len(df.columns)},
}


print(json.dumps(resultado, indent=2, ensure_ascii=False))

{
  "fase": 0,
  "verificacion": "sesion Spark + lectura CSV + escritura/lectura Parquet",
  "estado": "OK",
  "fecha_utc": "2026-11-21T00:22:47+00:00",
  "entorno": {
    "python": "3.14.4",
    "java": "21.0.13.1",
    "pyspark": "4.3.0",
    "pandas": "3.1.1",
    "pyarrow": "26.0.1",
    "spark_master": "local[*]",
    "paralelismo": 12,
    "driver_memory": "4g"
  },
  "csv_prueba": {
    "archivo": "data/_muestras/prueba_fase0.csv",
    "filas": 5,
    "columnas": 6
  }
}


In [8]:
spark.stop()